In [1]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 100]),
 'token_type_ids': torch.Size([32, 100]),
 'attention_mask': torch.Size([32, 100])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
teacher_model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [25]:
teacher_model.set_use_module_grafting(False)
teacher_model.set_use_scc_status(False)

### Load Student Model

In [26]:
load_path = f'./similar_layer_merge/weights/layer-6-{task_name}.pt'
history = torch.load(load_path, map_location='cpu')

In [27]:
from merge_helper import merge_bert_layers

In [28]:
for merge_layer, layer in enumerate(history['layer_track']):
    if len(layer) > 1:
        print("Merged Layer:", merge_layer)
        for _ in range(len(layer) -  1):
            merged = merge_bert_layers(model, merge_layer, merge_layer + 1)
            model.bert.encoder.layer[merge_layer] = merged
            del model.bert.encoder.layer[merge_layer + 1]

Merged Layer: 1
Merged Layer: 4
Merged Layer: 5


In [29]:
len(model.bert.encoder.layer)

6

In [30]:
# model.load_state_dict(history['model_state_dict'])

### Load Teacher Model

In [31]:
load_path = './glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location='cpu')
teacher_model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [32]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)
teacher_model.to(device)

device

device(type='cuda', index=1)

In [33]:
from train_eval_func import eval_loop

In [34]:
eval_loop(model, val_dataloader, task_name, device)

({'accuracy': 0.3161764705882353, 'f1': 0.0}, 1.1232194258616521)

In [35]:
eval_loop(teacher_model, val_dataloader, task_name, device)

({'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624},
 0.8802372308877798)

# 5. Layer Distillation

In [47]:
model.bert.encoder.layer[5]

BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (query): Linear(in_features=768, out_features=3072, bias=True)
      (key): Linear(in_features=768, out_features=3072, bias=True)
      (value): Linear(in_features=768, out_features=3072, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=3072, out_features=768, bias=True)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=12288, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): BertOutput(
    (dense): Linear(in_features=12288, out_features=768, bias=True)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [48]:
for batch in train_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)
    break

In [49]:
outputs.keys()

odict_keys(['loss', 'logits', 'hidden_states', 'attentions'])

In [50]:
outputs['hidden_states']['hidden_states'][5].shape

torch.Size([32, 89, 768])

In [51]:
for batch in train_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = teacher_model(**batch)
    break

In [52]:
outputs.keys()

odict_keys(['loss', 'logits', 'hidden_states', 'attentions'])

In [53]:
outputs['hidden_states']['hidden_states'][5].shape

torch.Size([32, 82, 768])

In [54]:
for batch in train_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    outputs = model(**batch)
    with torch.no_grad():
        outputs_teacher  = teacher_model(**batch)
    break

In [55]:
len(outputs['hidden_states']['hidden_states'])

7

In [56]:
outputs['hidden_states']['hidden_states'][6].shape

torch.Size([32, 77, 768])

In [57]:
len(outputs_teacher['hidden_states']['hidden_states'])

13

In [58]:
outputs_teacher['hidden_states']['hidden_states'][5].shape

torch.Size([32, 77, 768])

In [59]:
history['layer_track']

[[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]

In [60]:
student_layer_ids = [1,2,3,4,5,6]

In [61]:
teacher_layer_ids = []
for layer_idx in history['layer_track']:
    teacher_layer_ids.append(layer_idx[-1])

teacher_layer_ids

[0, 2, 3, 4, 7, 11]

In [62]:
import torch.nn.functional as F

In [63]:
hidden_loss = 0

for s, t in zip(student_layer_ids, teacher_layer_ids):

    student_hidden = outputs['hidden_states']['hidden_states'][s]
    teacher_hidden = outputs_teacher['hidden_states']['hidden_states'][t]

    hidden_loss += F.mse_loss(
        student_hidden,
        teacher_hidden
    )

hidden_loss /= len(student_layer_ids)

In [64]:
hidden_loss

tensor(0.2835, device='cuda:1', grad_fn=<DivBackward0>)

# 5. Learning Rate and Leanring Scheduler Setup

In [65]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=3e-5)

In [66]:
from train_eval_func import set_lr_scheduler

num_epochs = 10
num_training_steps = num_epochs * len(train_dataloader)

lr_scheduler = set_lr_scheduler(
    optimizer          = optimizer,
    num_training_steps = num_training_steps
)

print(num_training_steps)

1150


In [67]:
patience=3

# 6. Training

In [68]:
from train_eval_func import train

In [69]:
train_stats = train(
    model,
    train_dataloader,
    val_dataloader,
    task_name,
    device,
    num_epochs=num_epochs,
    lr=lr_scheduler,
    patience=patience,
    use_early_stopping=True,
    optimizer=optimizer,
    lr_scheduler=lr_scheduler,
    # save_path=save_path,
    display_epoch_iter=True,
    teacher_model=teacher_model,
    student_layer_ids=student_layer_ids,
    teacher_layer_ids=student_layer_ids
)

Training:   0%|          | 0/1150 [00:00<?, ?it/s]

Knowledge distillation used!


/home/abhinavl/work/Layer_Graft/layer-grafting-publication/train_eval_func.py:240: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  total_loss += float(loss)
Training:  10%|█         | 115/1150 [00:20<02:43,  6.34it/s]


<----------------- Epoch 1 ----------------->
Loss: 2.71, Training Metrics:
  accuracy: 0.6998
  f1: 0.8045
Validation Loss: 0.8, Validation Metrics:
  accuracy: 0.7475
  f1: 0.8418
Elapsed Time: 22.2147 sec


Training:  20%|██        | 230/1150 [00:44<03:24,  4.50it/s]

...


<----------------- Epoch 8 ----------------->
Loss: 0.15, Training Metrics:
  accuracy: 0.9875
  f1: 0.9907
Validation Loss: 0.95, Validation Metrics:
  accuracy: 0.8554
  f1: 0.9005
Elapsed Time: 24.6307 sec


Training:  90%|█████████ | 1035/1150 [03:33<00:34,  3.37it/s]


<----------------- Epoch 9 ----------------->
Loss: 0.13, Training Metrics:
  accuracy: 0.9875
  f1: 0.9907
Validation Loss: 0.89, Validation Metrics:
  accuracy: 0.8529
  f1: 0.8980
Elapsed Time: 26.5913 sec


Training: 100%|██████████| 1150/1150 [03:56<00:00,  4.74it/s]


<----------------- Epoch 10 ----------------->
Loss: 0.11, Training Metrics:
  accuracy: 0.9913
  f1: 0.9935
Validation Loss: 0.92, Validation Metrics:
  accuracy: 0.8529
  f1: 0.8990
Elapsed Time: 23.2222 sec

Total Training Time: 237.68 sec


In [70]:
eval_loop(model, eval_dataloader, task_name, device)

Training: 100%|██████████| 1150/1150 [04:04<00:00,  4.71it/s]


({'accuracy': 0.8243478260869566, 'f1': 0.8759721653704462},
 1.1360480208639745)

In [71]:
eval_loop(teacher_model, eval_dataloader, task_name, device)

({'accuracy': 0.8133333333333334, 'f1': 0.8686786296900489},
 1.1832497396411512)